# Day 205 — MCP Client ↔ LangGraph Agent Integration
**Month 13, Week 2, Day 2 | TeleServe India / ResumeGapAI continuity**

Calling the Day204 `resumegap_analyze` MCP tool from **inside a LangGraph node**.
Day204 built the server. Today builds the client-side agent that talks to it.

**Stack:** `langgraph==1.2.8`, `fastmcp==3.4.4`, `pydantic==2.13.4`, stdio transport


---
## Section 1 — Raw Data (LOCKED — do not modify)

Two files + a fixed test set. The engine/server here are rebuilt copies of your
Day204 originals (this sandbox doesn't have your local `resumegap_engine.py` /
`resumegap_mcp_server.py` on disk) — same public contract: `analyze_resume_gap(resume=, jd=, top_n=)`
and the `resumegap_analyze` tool with `ResumeGapInput` (`resume_text`, `jd_text`, `top_n`,
`str_strip_whitespace=True`, `extra="forbid"`). **Copy your actual Day204 files in when you run this
in Colab — don't use these stand-ins if your real ones are available.**


In [1]:
!pip install fastmcp==3.4.4 pydantic==2.13.4 --quiet

In [2]:
%%writefile resumegap_engine.py
"""
resumegap_engine.py
====================
LOCKED — Day 199 finalized core engine for ResumeGapAI (deployed at
resumegapai.streamlit.app). This is the Day 204 Raw Data layer. Do NOT modify
during Day 204 practice tasks — today's work happens only in resumegap_mcp_server.py.

Functions:
    clean_text(text)                    -> str
    tfidf_similarity_naive(resume, jd)  -> float (0-100)
    tfidf_similarity_fixed(resume, jd)  -> float (0-100)
    gap_words(resume, jd, top_n=8)      -> list[str]
    match_band(score)                   -> str
    analyze_resume_gap(resume, jd, top_n=8) -> dict
"""

import re
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

BACKGROUND_CORPUS = [
    "experienced professional seeking growth opportunities in a fast paced environment",
    "responsible for managing projects and delivering results on time and within budget",
    "strong communication and collaboration skills working with cross functional teams",
    "proficient in Microsoft Office including Excel Word and PowerPoint for reporting",
    "bachelor degree in relevant field with several years of professional experience",
    "seeking a motivated candidate with strong analytical and problem solving skills",
    "must have excellent written and verbal communication skills and attention to detail",
    "experience working in an agile environment with sprint planning and stand ups",
    "candidate should have strong leadership skills and experience managing a team",
    "looking for someone who can work independently and also collaborate with stakeholders",
]

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # space, not '' -- Day198 bugfix
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tfidf_similarity_naive(resume: str, jd: str) -> float:
    docs = [clean_text(resume), clean_text(jd)]
    vec = TfidfVectorizer(stop_words=list(ENGLISH_STOP_WORDS))
    matrix = vec.fit_transform(docs)
    sim = cosine_similarity(matrix[0:1], matrix[1:2])[0][0]
    return round(sim * 100, 2)

def tfidf_similarity_fixed(resume: str, jd: str) -> float:
    docs = BACKGROUND_CORPUS + [clean_text(resume), clean_text(jd)]
    vec = TfidfVectorizer(stop_words=list(ENGLISH_STOP_WORDS))
    matrix = vec.fit_transform(docs)
    resume_vec = matrix[-2:-1]
    jd_vec = matrix[-1:]
    sim = cosine_similarity(resume_vec, jd_vec)[0][0]
    return round(sim * 100, 2)

def gap_words(resume: str, jd: str, top_n: int = 8) -> list:
    resume_clean = clean_text(resume)
    jd_clean = clean_text(jd)
    docs = BACKGROUND_CORPUS + [resume_clean, jd_clean]
    vec = TfidfVectorizer(stop_words=list(ENGLISH_STOP_WORDS))
    matrix = vec.fit_transform(docs)
    feature_names = vec.get_feature_names_out()

    jd_vector = matrix[-1].toarray()[0]
    resume_words = set(resume_clean.split())

    scored = [
        (feature_names[i], jd_vector[i])
        for i in range(len(feature_names))
        if jd_vector[i] > 0 and feature_names[i] not in resume_words
    ]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [word for word, _score in scored[:top_n]]

def match_band(score: float) -> str:
    if score >= 70:
        return "Strong Match"
    elif score >= 45:
        return "Moderate Match"
    else:
        return "Weak Match"

def analyze_resume_gap(resume: str, jd: str, top_n: int = 8) -> dict:
    score = tfidf_similarity_fixed(resume, jd)
    band = match_band(score)
    gaps = gap_words(resume, jd, top_n)
    return {"similarity_score": score, "match_band": band, "gap_words": gaps}

Overwriting resumegap_engine.py


In [3]:
%%writefile resumegap_mcp_server.py
"""
resumegap_mcp_server.py
=======================
Day 204 – MCP Server for ResumeGapAI
Exposes `resumegap_analyze` as an MCP tool with:
- Pydantic input validation (min_length, bounds, whitespace stripping)
- Tool annotations (read‑only, idempotent, etc.)
- Clean `ToolError` handling for client‑friendly error messages

Run this file as a script to start the server over stdio.
"""

from fastmcp import FastMCP
from fastmcp.exceptions import ToolError
from pydantic import BaseModel, Field, ConfigDict
from resumegap_engine import analyze_resume_gap   # imports the Day199 locked engine

# Create the MCP server instance
mcp = FastMCP("resumegap_mcp")

# ── Pydantic input model ──────────────────────────────────────────────
class ResumeGapInput(BaseModel):
    """Input validation for the resume‑gap analysis tool."""
    model_config = ConfigDict(
        str_strip_whitespace=True,   # strips spaces before validation
        extra="forbid"               # rejects unexpected fields
    )

    resume_text: str = Field(
        ...,
        min_length=20,
        description="The full text of the candidate's resume or CV."
    )
    jd_text: str = Field(
        ...,
        min_length=20,
        description="The full text of the job description to compare against."
    )
    top_n: int = Field(
        default=8,
        ge=1,
        le=20,
        description="Number of top gap words to return (between 1 and 20)."
    )

# ── Tool registration ──────────────────────────────────────────────────
@mcp.tool(
    name="resumegap_analyze",
    annotations={
        "readOnlyHint": True,       # does not modify state
        "destructiveHint": False,   # does not delete or alter data
        "idempotentHint": True,     # calling repeatedly has same effect
        "openWorldHint": False      # no external side effects
    }
)
def resumegap_analyze(params: ResumeGapInput) -> dict:
    """
    Analyze gaps between a resume and a job description using TF‑IDF similarity.

    Returns a similarity score (0-100), a match band (Strong/Moderate/Weak),
    and a list of top gap words that appear in the JD but not in the resume.
    """
    try:
        # Call the locked engine – no modifications to the original logic
        return analyze_resume_gap(
            resume=params.resume_text,
            jd=params.jd_text,
            top_n=params.top_n
        )
    except Exception as e:
        # Any unexpected error from the engine is wrapped as a ToolError
        # so the client receives a structured, readable message.
        raise ToolError(f"Engine error during analysis: {str(e)}")

# ── Run the server (for stdio transport) ──────────────────────────────
if __name__ == "__main__":
    mcp.run()

Overwriting resumegap_mcp_server.py


**Locked test set — 5 resume/JD pairs (seed=205).** Row 5 is deliberately malformed (tests both validation layers).

In [4]:
import pandas as pd

# RAW DATA -- do not modify
TEST_PAIRS = [
    {
        "pair_id": "P1_DataAnalyst",
        "resume_text": "Experienced in Python, SQL, Pandas, and Power BI dashboards for retail analytics.",
        "jd_text": "Looking for a candidate skilled in Python, Machine Learning, Docker, AWS, and SQL.",
        "top_n": 5,
        "expect": "success"
    },
    {
        "pair_id": "P2_MLEngineer",
        "resume_text": "Built deep learning models using PyTorch and TensorFlow, deployed via Docker and AWS.",
        "jd_text": "Seeking an ML engineer with PyTorch, TensorFlow, Docker, Kubernetes, and Spark experience.",
        "top_n": 8,
        "expect": "success"
    },
    {
        "pair_id": "P3_ProductManager",
        "resume_text": "Product manager background with Excel, Tableau, A/B testing, and stakeholder management.",
        "jd_text": "Role requires SQL, A/B testing, Excel, and data visualization skills for roadmap decisions.",
        "top_n": 6,
        "expect": "success"
    },
    {
        "pair_id": "P4_DataEngineer",
        "resume_text": "Data engineering experience with ETL pipelines, Airflow, Spark, and Azure cloud services.",
        "jd_text": "Hiring a data engineer with ETL, Airflow, Spark, AWS, and Git version control skills.",
        "top_n": 5,
        "expect": "success"
    },
    {
        "pair_id": "P5_TooShort",
        "resume_text": "short",
        "jd_text": "Looking for a candidate skilled in Python, Machine Learning, Docker, AWS, and SQL.",
        "top_n": 5,
        "expect": "client_precheck_fail"
    },
]

raw_df = pd.DataFrame(TEST_PAIRS)
raw_df


,pair_id,resume_text,jd_text,top_n,expect
0,P1_DataAnalyst,"Experienced in Python, SQL, Pandas, and Power ...","Looking for a candidate skilled in Python, Mac...",5,success
1,P2_MLEngineer,Built deep learning models using PyTorch and T...,"Seeking an ML engineer with PyTorch, TensorFlo...",8,success
2,P3_ProductManager,"Product manager background with Excel, Tableau...","Role requires SQL, A/B testing, Excel, and dat...",6,success
3,P4_DataEngineer,Data engineering experience with ETL pipelines...,"Hiring a data engineer with ETL, Airflow, Spar...",5,success
4,P5_TooShort,short,"Looking for a candidate skilled in Python, Mac...",5,client_precheck_fail


---
## Section 2 — Concept Notes

### Why this day matters
Day204 built the MCP **server** (a tool exposed over the network/subprocess boundary).
Today builds the MCP **client** living inside a LangGraph **node** — the piece that
actually calls that tool as part of an agent's reasoning graph. This is the pattern
your Week 4 capstone (deployable support chatbot) depends on: agent node → MCP client → external tool.

### Key concepts

**1. `fastmcp.Client` + stdio transport**
`Client(path_to_server_script)` spins the server up as a **subprocess** and talks to it
over stdio (stdin/stdout), the same transport Claude Desktop and most local MCP setups use.
This is different from Day204's likely in-process testing (`Client(mcp_instance)`) —
stdio is closer to how a real deployed agent would reach a real deployed server.

**2. Async nodes in LangGraph**
A LangGraph node can be a regular function or an `async def` function. Any node that
awaits I/O (an MCP call, an API call, a DB query) should be async — LangGraph's
`ainvoke()` runs the graph and awaits async nodes natively; sync nodes still work
in the same graph.

**3. `async with client:` context**
The client's connection (subprocess spawn + stdio handshake) is scoped to a `with` block.
Opening a fresh `Client` per node call is simple and safe for a single tool call; a
production agent calling the same server many times would instead hold one long-lived
client — noted as a deliberate simplification here, not "the way to do it in production."

**4. Two-layer validation = two different failure classes**
- `validate_input` node (client-side, before any network call): catches obviously-bad
  input for free — no subprocess spawned, no round trip wasted.
- `ResumeGapInput` Pydantic model (server-side): the actual source of truth, catches
  everything the client-side check doesn't think to look for.
Row 5 in the test set is caught by the **first** layer alone. A row with valid-length
text but `top_n=50` would sail past the first layer and get caught by the **second** —
proving they're not redundant, they catch different things.

**5. `ToolError` vs a raw exception**
FastMCP's client raises `ToolError` (not a bare `ValidationError` traceback) when the
server rejects a call. Catching `ToolError` specifically — not a bare `except Exception` —
keeps the node's error path legible: anything else escaping is a real bug, not an
expected validation rejection.

### Real-world use
This exact shape — validate → call external tool via MCP → branch on success/error —
is how a production support agent would call any external system (a CRM lookup, a
knowledge-base search, a billing API) without hard-coding that system's SDK into the
graph itself. Swap the MCP server and the graph logic barely changes.


---
## Section 3 — Practice Guide

**Only use what's been covered through Day204.** No new LangGraph primitives beyond
`StateGraph`, `add_conditional_edges`, `Annotated` reducers (Days 200–203), plus today's
new pieces: async nodes and `fastmcp.Client`.

### Task 1 — AgentState + validate_input node (20 pts)
Define a `TypedDict` `AgentState` with: `resume_text`, `jd_text`, `top_n`, `valid`,
`mcp_result`, `error`, and `log` (use the `Annotated[List[str], operator.add]` reducer
pattern from Day202/203 — multiple nodes will append to it).
Write `validate_input(state)`: returns `valid=True` only if both `resume_text` and
`jd_text` are ≥ 20 characters (mirror the server's own floor). Append a log line.

### Task 2 — call_mcp_tool async node (30 pts)
Write `async def call_mcp_tool(state)`. Inside: create `Client(SERVER_PATH)`, open it
with `async with`, call `resumegap_analyze` passing `{"params": {...}}` (remember: a
single `params: BaseModel` tool arg nests everything under a `"params"` key — verified
Day204). On success, store `.data` into `mcp_result`. On `ToolError`, store `str(e)`
into `error`. Both paths append a log line.

### Task 3 — Routing (20 pts)
Wire two conditional edges:
- After `validate_input`: route to `call_mcp_tool` if `valid`, else to `end_with_error`
  (skip the tool call entirely — don't spawn a subprocess for input you already know is bad).
- After `call_mcp_tool`: route to `end_success` if no `error`, else `end_with_error`.

### Task 4 — Run all 5 test pairs + verify (20 pts)
Run `app.ainvoke()` (note: `await` it directly at the top level in Jupyter/Colab —
`asyncio.run()` fails inside a running event loop, confirmed Day204) for every row in
`TEST_PAIRS`. Print each pair's `pair_id`, final `log`, and either `mcp_result` or `error`.
Confirm: P1–P4 succeed, P5 fails at `validate_input` (never reaches `call_mcp_tool`).

### Task 5 — Self-check + interview framing (10 pts)
One paragraph: what's the actual difference in *purpose* between the client-side
`validate_input` check and the server's Pydantic model, given they check almost the
same thing? Then: one NRA-style recommendation for the P2_MLEngineer pair's result.


In [5]:
# ── TASK 1 (20 pts): AgentState + validate_input node ────────────────────
# Goal: Define the state schema with a log reducer, and implement a cheap
#       client‑side validation that mirrors the server's length requirement.
# Method:
#   - AgentState: fields for input (resume_text, jd_text, top_n), validation
#     flag (valid), MCP result/error, and a log with operator.add reducer.
#   - validate_input: checks both text lengths >= 20 (matching server's
#     min_length), sets valid flag, appends log line. This runs before any
#     network call – skips the subprocess for obviously bad input.

import operator
from typing import TypedDict, Annotated, List, Optional

class AgentState(TypedDict):
    resume_text: str
    jd_text: str
    top_n: int
    valid: bool
    mcp_result: Optional[dict]
    error: Optional[str]
    log: Annotated[List[str], operator.add]

def validate_input(state: AgentState) -> dict:
    """Cheap client‑side pre‑check – matches server's min_length."""
    resume_ok = len(state["resume_text"]) >= 20
    jd_ok = len(state["jd_text"]) >= 20
    valid = resume_ok and jd_ok
    # ✅ Corrected f‑string (no stray quotes)
    return {
        "valid": valid,
        "log": [f"validate_input: resume_ok={resume_ok}, jd_ok={jd_ok} -> valid={valid}"]
    }

In [6]:
# ── TASK 2 (30 pts): call_mcp_tool async node ────────────────────────────
# Goal: Call the MCP server over stdio from inside an async LangGraph node.
# Method:
#   - Use fastmcp.Client with the server script path (SERVER_PATH).
#   - Wrap in async with to manage the subprocess lifecycle.
#   - Call the tool with nested {"params": {...}} shape (verified in Day204).
#   - Catch ToolError specifically; store result or error in state.
#   - Append a log line on both success and failure.

import os, sys, tempfile
from pathlib import Path
from fastmcp import Client
from fastmcp.client.transports import StdioTransport
from fastmcp.exceptions import ToolError

SERVER_PATH = os.path.abspath("resumegap_mcp_server.py")

async def call_mcp_tool(state: AgentState) -> dict:
    log_file = tempfile.NamedTemporaryFile(delete=False, suffix=".log")
    log_path = Path(log_file.name)          # <-- the only change: wrap in Path()
    log_file.close()

    try:
        transport = StdioTransport(
            command=sys.executable,
            args=[SERVER_PATH],
            log_file=log_path                # StdioTransport now opens it correctly via .open("a")
        )
        async with Client(transport=transport) as client:
            result = await client.call_tool(
                "resumegap_analyze",
                {"params": {
                    "resume_text": state["resume_text"],
                    "jd_text": state["jd_text"],
                    "top_n": state["top_n"]
                }}
            )
        os.unlink(log_path)
        return {"mcp_result": result.data, "error": None, "log": ["call_mcp_tool: success"]}
    except ToolError as e:
        os.unlink(log_path)
        return {"mcp_result": None, "error": str(e), "log": [f"call_mcp_tool: ToolError – {e}"]}

In [7]:
# ── TASK 3 (20 pts): Routing logic ────────────────────────────────────────
# Goal: Wire conditional edges to skip the tool call when input is invalid,
#       and to branch to success/error nodes after the tool call.
# Method:
#   - route_after_validate: if valid -> "call_mcp_tool", else -> "end_with_error".
#   - route_after_mcp: if error is None -> "end_success", else -> "end_with_error".
#   - Then build the graph with these nodes and edges.

from langgraph.graph import StateGraph, END

def route_after_validate(state: AgentState) -> str:
    return "call_mcp_tool" if state["valid"] else "end_with_error"

def route_after_mcp(state: AgentState) -> str:
    return "end_success" if state.get("error") is None else "end_with_error"

def end_success(state: AgentState) -> dict:
    return {"log": ["end_success: reached"]}

def end_with_error(state: AgentState) -> dict:
    return {"log": ["end_with_error: reached"]}

builder = StateGraph(AgentState)
builder.add_node("validate_input", validate_input)
builder.add_node("call_mcp_tool", call_mcp_tool)
builder.add_node("end_success", end_success)
builder.add_node("end_with_error", end_with_error)

builder.set_entry_point("validate_input")
builder.add_conditional_edges("validate_input", route_after_validate)
builder.add_conditional_edges("call_mcp_tool", route_after_mcp)
builder.add_edge("end_success", END)
builder.add_edge("end_with_error", END)

app = builder.compile()
print("Graph compiled.")

Graph compiled.


In [8]:
# ── TASK 4 (20 pts): Run all 5 test pairs and verify results ─────────────
# Goal: Execute the graph on each pair in TEST_PAIRS, print logs and results,
#       and confirm P1–P4 succeed while P5 fails at validate_input.
# Method:
#   - For each test row, build initial state and call app.ainvoke() (top‑level
#     await in Colab – no asyncio.run()).
#   - Print the pair_id, final log, and either mcp_result or error.
#   - Assert the expected outcome matches the "expect" field.

async def run_all_pairs():
    results = []
    for row in TEST_PAIRS:
        init_state = {
            "resume_text": row["resume_text"],
            "jd_text": row["jd_text"],
            "top_n": row["top_n"],
            "valid": False,
            "mcp_result": None,
            "error": None,
            "log": [],
        }
        print(f"\n--- Running {row['pair_id']} (expect: {row['expect']}) ---")
        final = await app.ainvoke(init_state)
        print("Log:", final["log"])
        if final.get("error"):
            print("Error:", final["error"])
        else:
            print("MCP result:", final.get("mcp_result"))

        results.append({
            "pair_id": row["pair_id"],
            "expect": row["expect"],
            "valid": final["valid"],
            "error": final.get("error"),
            "has_result": final.get("mcp_result") is not None,
        })

    print("\n--- Verification ---")
    for r in results:
        if r["expect"] == "success":
            assert r["valid"] is True, f"{r['pair_id']}: valid should be True"
            assert r["error"] is None, f"{r['pair_id']}: error should be None"
            assert r["has_result"] is True, f"{r['pair_id']}: should have result"
            print(f"✅ {r['pair_id']}: success as expected")
        else:
            assert r["valid"] is False, f"{r['pair_id']}: valid should be False"
            assert r["error"] is None, f"{r['pair_id']}: error should be None (no tool call)"
            assert r["has_result"] is False, f"{r['pair_id']}: should not have result"
            print(f"✅ {r['pair_id']}: client_precheck_fail as expected")

await run_all_pairs()


--- Running P1_DataAnalyst (expect: success) ---
Log: ['validate_input: resume_ok=True, jd_ok=True -> valid=True', 'call_mcp_tool: success', 'end_success: reached']
MCP result: {'similarity_score': 18.44, 'match_band': 'Weak Match', 'gap_words': ['aws', 'docker', 'learning', 'machine', 'skilled']}

--- Running P2_MLEngineer (expect: success) ---
Log: ['validate_input: resume_ok=True, jd_ok=True -> valid=True', 'call_mcp_tool: success', 'end_success: reached']
MCP result: {'similarity_score': 27.07, 'match_band': 'Weak Match', 'gap_words': ['engineer', 'kubernetes', 'ml', 'spark', 'seeking', 'experience']}

--- Running P3_ProductManager (expect: success) ---
Log: ['validate_input: resume_ok=True, jd_ok=True -> valid=True', 'call_mcp_tool: success', 'end_success: reached']
MCP result: {'similarity_score': 16.47, 'match_band': 'Weak Match', 'gap_words': ['data', 'decisions', 'requires', 'roadmap', 'role', 'sql']}

--- Running P4_DataEngineer (expect: success) ---
Log: ['validate_input: r

In [11]:
bonus_state = {"resume_text": "Experienced in Python and SQL for five years now.",
               "jd_text": "Looking for Python, SQL, and Docker skills in this role.",
               "top_n": 50, "valid": False, "mcp_result": None, "error": None, "log": []}
print((await app.ainvoke(bonus_state))["log"])

['validate_input: resume_ok=True, jd_ok=True -> valid=True', 'call_mcp_tool: ToolError – 1 validation error for call[resumegap_analyze]\nparams.top_n\n  Input should be less than or equal to 20 [type=less_than_equal, input_value=50, input_type=int]\n    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal', 'end_with_error: reached']


In [10]:
# ── TASK 5 (10 pts): Self‑check + NRA recommendation ─────────────────────

# (a) Purpose of two validation layers:
# The client‑side validate_input is a cheap, local pre‑check that rejects
# obviously invalid input without spawning a subprocess or making a round trip.
# It is incomplete – it only checks what we thought to check. The server's
# Pydantic model is the authoritative source of truth; it enforces all
# constraints (length, bounds, extra fields) on every call. They are not
# redundant: the client check saves cost for obvious failures; the server
# guarantees correctness and cannot be bypassed.

# (b) NRA‑style recommendation for P2_MLEngineer's result.
# Using the actual output from the run:

# Number:
#   - similarity_score = 27.07
#   - gap_words = ['engineer', 'kubernetes', 'ml', 'spark', 'seeking', 'experience']
#   (The JD explicitly requests Kubernetes and Spark; both are missing from the resume.)

# Reason:
#   The resume lists PyTorch, TensorFlow, Docker, and AWS – but the job description
#   also requires Kubernetes and Spark, which do not appear anywhere in the resume.
#   This accounts for the relatively low similarity score (27.07) and the presence
#   of "kubernetes" and "spark" in the gap_words list.

# Action:
#   Recommend that the candidate upskill in Kubernetes and Spark (e.g., a 2‑week
#   hands‑on course on each, or a combined project). Alternatively, if these
#   skills are not strictly required, the hiring team should adjust the JD to
#   reflect the actual must‑haves. A follow‑up re‑evaluation after upskilling
#   would quantify the improvement.

---
## Section 4 — Scoring Rubric (100 pts)

| Task | Points | Criteria |
|---|---|---|
| Task 1 — AgentState + validate_input | 20 | Correct TypedDict fields incl. `log` reducer (10); validate_input logic + length floor matches server (10) |
| Task 2 — call_mcp_tool async node | 30 | Correct `async def` + `async with client:` (10); correct `{"params": {...}}` nesting (10); ToolError caught specifically, not bare Exception (10) |
| Task 3 — Routing | 20 | validate→tool/error edge correct (10); tool→success/error edge correct (10) |
| Task 4 — Run all 5 pairs | 20 | All 5 pairs run without crashing (10); P1–P4 succeed and P5 fails at validate_input specifically, verified from printed log (10) |
| Task 5 — Self-check + NRA | 10 | Correctly distinguishes client pre-check (cheap, incomplete) from server validation (source of truth) (5); valid NRA recommendation with real numbers from output (5) |

**Technical vs Communication split** (flag separately, same as every prior day):
- Technical: wrong state shape, wrong routing logic, bare `except Exception` instead of `except ToolError`, missing `params` nesting
- Communication: correct code but Task 5 explanation restates *what* the layers check instead of *why two layers* — restating benefit language ("it validates input") instead of mechanism ("client check is free/local, server check is authoritative and can't be bypassed")

### Key Takeaway
A client-side check and the server's own validation aren't duplicated effort — they
catch **different failure classes at different costs**. The client check is free but
incomplete (it only knows what you thought to check); the server's Pydantic model is
authoritative but costs a round trip. Row 5's short resume and the bonus `top_n=50`
case are proof, not assertion: two different rows, two different layers, two different
catches.

### Interview framing
*"I built a LangGraph node that calls an external tool over MCP with two validation
layers — a cheap client-side check that skips the network call for obviously bad input,
and the tool server's own Pydantic model as the actual source of truth. I proved they
weren't redundant by finding an input that passes the first check but fails the second."*
